In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')

(Tutorial_Append_structures)=
# Append structures

*Appending structures from one molecular system into another.*

Structures (frames or conformations) from different molecular systems can be appended to a given system using {func}`molsysmt.basic.append_structures`. In this way, an existing molecular system can be extended with additional structures, for example when combining snapshots from different simulations. The source can be a coordinate-only trajectory such as XTC, DCD, or XYZ; it does not need to repeat the target topology. Every stored structural series covers the complete resulting structure axis. By default, `attribute_policy='intersection'` discards optional series that are not available in both blocks and emits one warning; use `attribute_policy='strict'` to reject the operation without modifying the target instead.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown
Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.basic.append_structures`.
:::


## Basic usage

Let's show how this method works with alanine dipeptide defined as three different molecular systems with different structures.


In [2]:
import molsysmt as msm

In [3]:
molsys_A = msm.build.build_peptide('AceAlaNme')
molsys_B = msm.structure.translate(molsys_A, translation='[0.1, 0.1, 0.1] nanometers')
molsys_C = msm.structure.translate(molsys_A, translation='[0.2, 0.2, 0.2] nanometers')

In [4]:
msm.info(molsys_A)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,22,3,1,1,1,1,1,1


Now let's append the structures of $B$ and $C$ into $A$:

In [5]:
msm.append_structures(molsys_A, molsys_B)
msm.append_structures(molsys_A, molsys_C)

:::{tip}
:class: dropdown

All methods defined in the {ref}`molsysmt.basic <API basic>` module can be invoked also from the main level of the library. As such, {func}`molsysmt.append_structures` is the same method as {func}`molsysmt.basic.append_structures`.
:::

Let's see now the new content of $A$. We expect $A$ to now contain 3 structures:

In [6]:
msm.info(molsys_A)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,22,3,1,1,1,1,1,3


And we can also visualize it interactively. Try using the widget buttons to cycle through the 3 frames.

In [7]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

molsysviewer_htmlfile = '_static/views/tools_basic_append_structures.html'

In [8]:
msm.view(molsys_A)

'<iframe src="../../../../_static/views/tools_basic_append_structures.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

By default, the function modifies the target system in place. If you prefer not to modify the original molecular system, setting `in_place=False` returns a new molecular system with the appended structures, while leaving the original system unchanged:

In [9]:
molsys_D = msm.append_structures(molsys_B, molsys_C, in_place=False)

In [10]:
msm.get(molsys_B, n_structures=True)

1

In [11]:
msm.get(molsys_C, n_structures=True)

1

In [12]:
msm.get(molsys_D, n_structures=True)

2

## Appending selected structures

When the source system contains multiple structures, you can select specific frames to append using `structure_indices`. For example, let's concatenate $A$, $B$, and $C$ into a 3-structure system, and then append only structure `1` into $B$:

In [13]:
molsys_multi = msm.concatenate_structures([molsys_A, molsys_B, molsys_C])
molsys_E = msm.append_structures(molsys_B, molsys_multi, structure_indices=1, in_place=False)

In [14]:
msm.get(molsys_E, n_structures=True)

2

## Appending structural subsets

If the source system contains additional atoms or you want to append structures considering only a specific subset of atoms, pass a matching `selection` string:

In [15]:
molsys_A_C = msm.extract(molsys_A, selection='atom_type=="C"')
molsys_F = msm.append_structures(molsys_A_C, molsys_B, selection='atom_type=="C"', in_place=False)

In [16]:
msm.info(molsys_F)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,6,3,1,1,1,1,1,4


### Appending native observables

Temperature, potential energy and kinetic energy can be supplied directly to
native `Structures` construction and `append()` as unit-bearing series of shape
`(n_structures,)`. Each value belongs to one structure. A scalar becomes a
length-one series; it is not broadcast across a longer coordinate block.
Missing observables remain `None`, and an empty series represents zero structures.

Compatible units are converted on storage (K and kJ/mol); `get()` applies your
active output unit policy. Bare values, Boolean/complex values, matrices and
incompatible units are rejected. Series lengths must match the same coordinate
or metadata structure axis. Invalid append units, ranks or lengths leave the
object unchanged. Existing append policies apply: `intersection` keeps shared
attributes, while `strict` refuses one-sided attributes.

These observables survive supported `StructuresDict` and H5MSM 0.5 round trips.
MolSysDict 0.1 and the current structures YAML codec do not store these three
thermodynamic series. Converting to those formats does not extend their schema.
Scalar temperatures in simulation settings keep their existing contract.
See the {ref}`native input reference <API Native Structure Inputs>`.


In [1]:
import numpy as np
import molsysmt as msm
from molsysmt.native import Structures
puw = msm.pyunitwizard
molsys = Structures(
    coordinates=puw.quantity(np.zeros((2, 2, 3)), 'nm'),
    temperature=puw.quantity([300000., 301000.], 'mK'),
    potential_energy=puw.quantity([-1., -2.], 'kcal/mol'),
    kinetic_energy=puw.quantity([0.1, 0.2], 'kcal/mol'))
molsys.append(
    coordinates=puw.quantity(np.zeros((1, 2, 3)), 'nm'),
    temperature=puw.quantity(302., 'K'),
    potential_energy=puw.quantity(-3., 'kcal/mol'),
    kinetic_energy=puw.quantity(0.3, 'kcal/mol'),
    attribute_policy='strict')
selected = molsys.get(structure_indices=[2, 0, 2], temperature=True)
np.testing.assert_allclose(puw.get_value(selected, to_unit='K'), [302., 300., 302.])
stored = msm.convert(molsys, to_form='molsysmt.StructuresDict')
restored = msm.convert(stored, to_form='molsysmt.Structures')
np.testing.assert_allclose(
    puw.get_value(restored.kinetic_energy, to_unit='kJ/mol'), [0.4184, 0.8368, 1.2552])
print(puw.get_value(selected, to_unit='K').tolist())
print(molsys.n_structures)


[302.0, 300.0, 302.0]
3


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Tutorial_Build_peptide`: Build natural peptides with or without terminal caps with {func}`molsysmt.build.build_peptide`.
- {ref}`Tutorial_Translate`: Translate molecular systems in space with {func}`molsysmt.structure.translate`.
- {ref}`Tutorial_Info`: Print a summary of the contents, topology, and structural data of a molecular system with {func}`molsysmt.basic.info`.
- {ref}`Tutorial_View`: Show a molecular system interactively in 3D with {func}`molsysmt.basic.view`.
- {ref}`Tutorial_Get`: Retrieve attribute values from a molecular system with {func}`molsysmt.basic.get`.
- {ref}`Tutorial_Concatenate_structures`: Concatenate the structures found in a list of molecular systems with {func}`molsysmt.basic.concatenate_structures`.
- {ref}`Tutorial_Extract`: Extract a subset of atoms or structures from a molecular system into a new system with {func}`molsysmt.basic.extract`.
:::